# 稀疏矩阵存储格式优化

稀疏矩阵向量乘法（SpMV），即y=A×x，是科学计算、图计算和人工智能计算中的典型基础运算，其计算性能在很大程度上取决于稀疏矩阵的存储格式、数据分块方式和访存效率。在并行计算领域中，合理选择稀疏矩阵存储格式是优化矩阵计算性能的重要手段之一，通过压缩零元素、优化索引组织和并行任务划分，能够减少数据存储量和不必要的数据访问，提高访存带宽利用率和并行计算效率。

本实验围绕SpMV展开，基于静态Tensor编程范式实现。实验在Ascend C环境下，分别采用COO和CSR两种格式组织稀疏矩阵数据，并对比两种格式在昇腾NPU上的并行计算性能。其中，COO实现按照非零元素数量进行均衡分块，CSR实现按照矩阵行进行分块，由多个AI Core并行完成非零元素乘法与行内归约。

本节学习大纲如下：

1. 实验概述：介绍实验目标、前置知识和实验要点；
2. 环境准备：创建实验目录并加载CANN环境；
3. 问题分析：分析输入输出、COO与CSR数据布局、存储开销、并行划分方法和实验参数设置；
4. 核函数开发：实现COO分段计算、COO部分结果归约和CSR按行计算；
5. 结果验证与性能分析：准备Host侧输入数据和CPU参考结果，完成工程构建、算子运行、结果验证和性能分析；
6. 实验总结：归纳稀疏矩阵存储格式、分块计算、跨核归约和多核并行在SpMV中的实现过程。

---
## 1. 实验概述

本实验以SpMV为问题背景，在Ascend C环境下，以COO与CSR两种常用的矩阵存储格式为例，研究稀疏矩阵存储格式对算子性能的影响。实验中，分别按照非零元数量和矩阵行划分AI Core任务，比较两种数据组织与分块方式在存储开销、访存连续性、结果归约和并行执行效率方面的差异。

### 1.1 实验目标

完成本实验后应达到以下目标：

1. 掌握稀疏矩阵的存储格式。能够掌握COO和CSR两种存储格式的数据组织方式及特点，即COO显式保存行列索引、CSR利用行指针描述行边界的表示方法；能够比较两种格式在存储开销、连续访问和行归约方面的差异；能够根据实际问题选择更优的存储格式。
2. 理解SpMV的并行实现方法。能够根据COO与CSR的结构特点，分别采用按非零元素均分和按行划分组织AI Core任务，理解索引访问、局部乘法、行内归约及核间结果合并的基本方法。
3. 具备正确性验证和性能分析能力。能够在Host侧计算CPU基线实验结果，通过最大绝对误差和最大相对误差验证COO与CSR并行计算结果的正确性；能够通过总执行时间和加速比评价两种实现的性能差异。

### 1.2 前置知识

本实验要求提前具备以下基础：

1. SpMV计算基础：理解SpMV的数学含义和基本计算过程，明确输出向量中每个元素由矩阵对应行的非零元素与向量元素乘积累加得到，并认识SpMV以访存为主的计算特点。
2. 稀疏矩阵存储格式基础：理解COO和CSR两种稀疏矩阵存储格式的数据组织方式，即COO使用行索引、列索引和非零值三个数组记录非零元素，CSR使用行指针、列索引和非零值三个数组描述各行非零元素的存储范围。掌握COO按非零元素数量划分任务和CSR按矩阵行划分任务的基本方法，认识两种格式在访存连续性、负载均衡和结果合并方面的差异。
3. 数据分块与归约基础：理解按非零元素均分与按行划分两种任务组织方式，能够分析负载均衡、核间写冲突和归约开销之间的关系；明确Host侧负责数据准备与结果校验，Device侧负责AI Core并行计算。
4. Ascend C开发基础：Host侧负责输入数据准备、运行时管理、Device侧内存申请、Kernel启动和结果校验；Device侧执行Kernel核函数，完成SpMV的核心计算。实验前应熟悉工程编译与脚本运行方法，同时掌握性能指标查看方法。
5. 静态Tensor编程基础：理解静态Tensor编程方式要求开发者显式规划数据块边界、LocalTensor地址和存储位置，并根据数据搬入、计算和写回之间的依赖关系管理同步。理解数据通常先从全局内存搬入局部内存，再在片上完成计算，最后写回全局内存。

### 1.3 实验要点

实验中应重点关注以下内容：

1. 数据组织：分别使用COO与CSR组织同一个稀疏矩阵，明确行列索引与行指针对非零元分布的表达方式；
2. 数据分块：COO按非零元数量均衡分块，CSR按矩阵行范围分块，分析两种方法对负载均衡、访存连续性和结果合并的影响；
3. SpMV并行计算：在AI Core上完成稀疏矩阵数据搬运、输入向量索引读取、非零元乘法和行内求和，理解COO局部结果归约与CSR按行独占计算的差异；
4. 结果验证与性能分析：使用最大绝对误差、最大相对误差和不匹配元素数量验证结果，使用平均执行时间与COO和CSR时间比值分析性能。

---
## 2. 环境准备

### 2.1 创建实验目录并加载CANN环境

本小节创建实验所需目录，并加载Ascend CANN环境变量。

目录划分如下：

- `src/02.03_extra_spmv_coo_csr_storage_format/include`：保存Host侧公共头文件、问题配置和CPU接口定义。
- `src/02.03_extra_spmv_coo_csr_storage_format/src`：保存稀疏矩阵构造、CPU参考实现和CPU演示程序。
- `src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel`：保存Device侧核函数代码，包括COO分段计算、COO归约和CSR按行计算。
- `src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/host_launch`：保存Host侧核函数调用代码。
- `src/02.03_extra_spmv_coo_csr_storage_format/scripts`：保存构建与运行脚本。
- `src/02.03_extra_spmv_coo_csr_storage_format/results`：保存实验结果。

如果当前环境已安装CANN，下面的代码会加载`set_env.sh`。如果没有找到该文件，仍然可以继续查看和生成代码，但完整编译和运行需要在已配置CANN的环境中完成。

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

WORK_DIR = Path("src/02.03_extra_spmv_coo_csr_storage_format").resolve()
SRC_DIR = WORK_DIR / "src"

# 创建本实验目录结构
(WORK_DIR / "results").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "scripts").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "include").mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)
(WORK_DIR / "ascend_ops" / "op_kernel").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "ascend_ops" / "host_launch").mkdir(parents=True, exist_ok=True)

# 自动查找并加载CANN环境
arch = os.uname().machine
candidate_paths = []

for name in ["ASCEND_INSTALL_PATH", "ASCEND_TOOLKIT_HOME"]:
    value = os.environ.get(name)
    if value:
        candidate_paths.append(Path(value))

ascend_home = os.environ.get("ASCEND_HOME_PATH")
if ascend_home:
    candidate_paths.append(Path(ascend_home) / f"{arch}-linux")

candidate_paths.extend(
    sorted(Path("/opt/conda/Ascend").glob(f"cann-*/{arch}-linux"), reverse=True)
)
candidate_paths.append(Path("/usr/local/Ascend/ascend-toolkit/latest"))
candidate_paths.extend(
    sorted(Path("/usr/local/Ascend/ascend-toolkit").glob(f"*/{arch}-linux"), reverse=True)
)

set_env = None
for item in candidate_paths:
    if (item / "set_env.sh").exists():
        set_env = item / "set_env.sh"
        break

if set_env is not None:
    command = f"source {shlex.quote(str(set_env))} && env"
    env = subprocess.check_output(["bash", "-lc", command], text=True)
    for line in env.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value
    os.environ["ASCEND_INSTALL_PATH"] = str(set_env.parent)
    print("Ascend environment loaded from:", set_env)
else:
    print("Ascend set_env.sh was not found. Build and run need CANN.")

print("Experiment directory:", WORK_DIR)
print("Source directory:", SRC_DIR)


### 2.2 写入工程公共文件

本节写入公共头文件、CPU参考实现和Host侧启动代码。Device侧核函数将在第4节写入，CMake构建文件和运行脚本将在第5节写入。运行到第5节后，`src/02.03_extra_spmv_coo_csr_storage_format`目录下将形成可直接编译执行的Ascend C工程。

In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/include/spmv_common.h
#pragma once

#include <cstddef>
#include <cstdint>
#include <string>
#include <vector>

namespace spmv {

constexpr uint32_t kDefaultRows = 2048;
constexpr uint32_t kDefaultCols = 2048;
constexpr uint32_t kDefaultNnzPerRow = 32;
constexpr uint32_t kDefaultBlockDim = 32;
constexpr uint32_t kAlignmentElements = 8;  // 32 bytes for float/uint32_t.

struct Config {
    uint32_t rows = kDefaultRows;
    uint32_t cols = kDefaultCols;
    uint32_t nnz_per_row = kDefaultNnzPerRow;
    uint32_t block_dim = kDefaultBlockDim;
    uint32_t seed = 20260717;
    uint32_t warmup = 5;
    uint32_t repeat = 30;
};

struct CooMatrix {
    std::vector<float> values;
    std::vector<uint32_t> rows;
    std::vector<uint32_t> cols;
};

struct CsrMatrix {
    std::vector<float> values;
    std::vector<uint32_t> col_indices;
    std::vector<uint32_t> row_ptr;
};

struct Problem {
    Config config;
    CooMatrix coo;
    CsrMatrix csr;
    std::vector<float> x;
};

struct ErrorMetrics {
    double max_abs = 0.0;
    double max_rel = 0.0;
    size_t mismatch_count = 0;
};

struct TimedResult {
    std::vector<float> y;
    double elapsed_us = 0.0;
};

uint32_t align_up(uint32_t value, uint32_t alignment = kAlignmentElements);
uint32_t div_up(uint32_t numerator, uint32_t denominator);

void validate_config(const Config& config, bool for_ascend);
Problem make_problem(const Config& config);
std::vector<float> spmv_reference(const Problem& problem);

TimedResult run_coo_cpu_partitioned(const Problem& problem);
TimedResult run_csr_cpu_partitioned(const Problem& problem);
ErrorMetrics compare_vectors(const std::vector<float>& actual,
                             const std::vector<float>& expected,
                             float tolerance = 1.0e-4F);

size_t coo_storage_bytes(const Problem& problem);
size_t csr_storage_bytes(const Problem& problem);
void print_problem_summary(const Problem& problem);
void print_output_sample(const std::vector<float>& y,
                         const std::vector<float>& reference,
                         size_t count = 8);

}  // namespace spmv


In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/include/spmv_cpu.h
#pragma once

#include "spmv_common.h"

namespace spmv {

TimedResult benchmark_coo_cpu(const Problem& problem, uint32_t warmup, uint32_t repeat);
TimedResult benchmark_csr_cpu(const Problem& problem, uint32_t warmup, uint32_t repeat);

}  // namespace spmv


In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/src/spmv_cpu.cpp
#include "spmv_cpu.h"

#include <algorithm>
#include <chrono>
#include <cmath>
#include <iomanip>
#include <iostream>
#include <limits>
#include <numeric>
#include <random>
#include <stdexcept>
#include <unordered_set>

namespace spmv {
namespace {

// 使用统一计时器记录各实现单次运行的微秒级耗时。
class Timer {
public:
    Timer() : begin_(std::chrono::high_resolution_clock::now()) {}

    double elapsed_us() const {
        const auto end = std::chrono::high_resolution_clock::now();
        return std::chrono::duration<double, std::micro>(end - begin_).count();
    }

private:
    std::chrono::high_resolution_clock::time_point begin_;
};

}  // namespace

uint32_t align_up(uint32_t value, uint32_t alignment) {
    if (alignment == 0) {
        throw std::invalid_argument("alignment must be positive");
    }
    return div_up(value, alignment) * alignment;
}

uint32_t div_up(uint32_t numerator, uint32_t denominator) {
    if (denominator == 0) {
        throw std::invalid_argument("denominator must be positive");
    }
    return (numerator + denominator - 1) / denominator;
}

// 校验公共实验参数，并在NPU模式下检查静态张量容量和DataCopy对齐约束。
void validate_config(const Config& config, bool for_ascend) {
    if (config.rows == 0 || config.cols == 0 || config.nnz_per_row == 0) {
        throw std::invalid_argument("rows, cols, and nnz-per-row must all be positive");
    }
    if (config.nnz_per_row > config.cols) {
        throw std::invalid_argument("nnz-per-row cannot exceed cols");
    }
    if (config.block_dim == 0) {
        throw std::invalid_argument("block-dim must be positive");
    }
    if (config.repeat == 0) {
        throw std::invalid_argument("repeat must be positive");
    }
    if (for_ascend) {
        if (config.rows > 4096 || config.cols > 4096) {
            throw std::invalid_argument("Ascend C static tensors support rows/cols up to 4096");
        }
        if ((config.rows % kAlignmentElements) != 0 ||
            (config.cols % kAlignmentElements) != 0 ||
            (config.nnz_per_row % kAlignmentElements) != 0) {
            throw std::invalid_argument(
                "for Ascend C, rows, cols, and nnz-per-row must be multiples of 8 "
                "so every DataCopy is 32-byte aligned");
        }
    }
}

// 使用同一组非零元素同时构造COO与CSR数据，保证两种格式的输入完全一致。
Problem make_problem(const Config& config) {
    validate_config(config, false);

    Problem problem;
    problem.config = config;
    const uint64_t nnz64 = static_cast<uint64_t>(config.rows) * config.nnz_per_row;
    if (nnz64 > std::numeric_limits<uint32_t>::max()) {
        throw std::invalid_argument("matrix has too many non-zero elements for uint32_t indices");
    }
    const size_t nnz = static_cast<size_t>(nnz64);
    problem.coo.values.reserve(nnz);
    problem.coo.rows.reserve(nnz);
    problem.coo.cols.reserve(nnz);
    problem.csr.values.reserve(nnz);
    problem.csr.col_indices.reserve(nnz);
    problem.csr.row_ptr.resize(static_cast<size_t>(config.rows) + 1, 0);
    problem.x.resize(config.cols);

    std::mt19937 generator(config.seed);
    std::uniform_real_distribution<float> value_dist(-1.0F, 1.0F);
    std::uniform_int_distribution<uint32_t> column_dist(0, config.cols - 1);
    for (float& value : problem.x) {
        value = value_dist(generator);
    }

    for (uint32_t row = 0; row < config.rows; ++row) {
        std::unordered_set<uint32_t> unique_columns;
        unique_columns.reserve(static_cast<size_t>(config.nnz_per_row) * 2);
        while (unique_columns.size() < config.nnz_per_row) {
            unique_columns.insert(column_dist(generator));
        }
        std::vector<uint32_t> row_columns(unique_columns.begin(), unique_columns.end());
        std::sort(row_columns.begin(), row_columns.end());

        problem.csr.row_ptr[row] = static_cast<uint32_t>(problem.csr.values.size());
        for (uint32_t col : row_columns) {
            const float value = value_dist(generator);
            problem.coo.values.push_back(value);
            problem.coo.rows.push_back(row);
            problem.coo.cols.push_back(col);
            problem.csr.values.push_back(value);
            problem.csr.col_indices.push_back(col);
        }
    }
    problem.csr.row_ptr[config.rows] = static_cast<uint32_t>(problem.csr.values.size());
    return problem;
}

// 按COO非零元素顺序串行累加，生成正确性验证所需的CPU参考结果。
std::vector<float> spmv_reference(const Problem& problem) {
    std::vector<float> y(problem.config.rows, 0.0F);
    for (size_t i = 0; i < problem.coo.values.size(); ++i) {
        y[problem.coo.rows[i]] += problem.coo.values[i] * problem.x[problem.coo.cols[i]];
    }
    return y;
}

// 模拟COO两阶段并行流程：先按非零元素分块计算，再按行归约各块结果。
TimedResult run_coo_cpu_partitioned(const Problem& problem) {
    const Config& config = problem.config;
    const uint32_t stride = align_up(config.rows);
    const uint32_t nnz_per_core = align_up(
        div_up(static_cast<uint32_t>(problem.coo.values.size()), config.block_dim));

    Timer timer;
    std::vector<float> partial(static_cast<size_t>(config.block_dim) * stride, 0.0F);
    for (uint32_t core = 0; core < config.block_dim; ++core) {
        const uint32_t begin = core * nnz_per_core;
        const uint32_t end = std::min<uint32_t>(
            static_cast<uint32_t>(problem.coo.values.size()), begin + nnz_per_core);
        float* const core_partial = partial.data() + static_cast<size_t>(core) * stride;
        for (uint32_t i = begin; i < end; ++i) {
            core_partial[problem.coo.rows[i]] +=
                problem.coo.values[i] * problem.x[problem.coo.cols[i]];
        }
    }

    // 第二阶段汇总各逻辑核的部分向量，消除跨块行冲突。
    std::vector<float> y(config.rows, 0.0F);
    for (uint32_t row = 0; row < config.rows; ++row) {
        float sum = 0.0F;
        for (uint32_t core = 0; core < config.block_dim; ++core) {
            sum += partial[static_cast<size_t>(core) * stride + row];
        }
        y[row] = sum;
    }
    return {std::move(y), timer.elapsed_us()};
}

// 模拟CSR按行分块流程，每个逻辑核独占一段输出行，无需跨核归约。
TimedResult run_csr_cpu_partitioned(const Problem& problem) {
    const Config& config = problem.config;
    const uint32_t rows_per_core = align_up(div_up(config.rows, config.block_dim));

    Timer timer;
    std::vector<float> y(config.rows, 0.0F);
    for (uint32_t core = 0; core < config.block_dim; ++core) {
        const uint32_t row_begin = core * rows_per_core;
        const uint32_t row_end = std::min(config.rows, row_begin + rows_per_core);
        for (uint32_t row = row_begin; row < row_end; ++row) {
            float sum = 0.0F;
            for (uint32_t i = problem.csr.row_ptr[row]; i < problem.csr.row_ptr[row + 1]; ++i) {
                sum += problem.csr.values[i] * problem.x[problem.csr.col_indices[i]];
            }
            y[row] = sum;
        }
    }
    return {std::move(y), timer.elapsed_us()};
}

// 预热阶段不计时，正式阶段累计多次结果并计算平均执行时间。
TimedResult benchmark_coo_cpu(const Problem& problem, uint32_t warmup, uint32_t repeat) {
    for (uint32_t i = 0; i < warmup; ++i) {
        (void)run_coo_cpu_partitioned(problem);
    }
    TimedResult result;
    for (uint32_t i = 0; i < repeat; ++i) {
        TimedResult current = run_coo_cpu_partitioned(problem);
        result.elapsed_us += current.elapsed_us;
        result.y = std::move(current.y);
    }
    result.elapsed_us /= repeat;
    return result;
}

TimedResult benchmark_csr_cpu(const Problem& problem, uint32_t warmup, uint32_t repeat) {
    for (uint32_t i = 0; i < warmup; ++i) {
        (void)run_csr_cpu_partitioned(problem);
    }
    TimedResult result;
    for (uint32_t i = 0; i < repeat; ++i) {
        TimedResult current = run_csr_cpu_partitioned(problem);
        result.elapsed_us += current.elapsed_us;
        result.y = std::move(current.y);
    }
    result.elapsed_us /= repeat;
    return result;
}

// 计算最大绝对误差、最大相对误差及超过容差的元素数量。
ErrorMetrics compare_vectors(const std::vector<float>& actual,
                             const std::vector<float>& expected,
                             float tolerance) {
    if (actual.size() != expected.size()) {
        throw std::invalid_argument("cannot compare vectors with different lengths");
    }
    ErrorMetrics metrics;
    for (size_t i = 0; i < actual.size(); ++i) {
        const double abs_error = std::abs(static_cast<double>(actual[i]) - expected[i]);
        const double rel_error = abs_error / std::max(std::abs(static_cast<double>(expected[i])), 1.0e-12);
        metrics.max_abs = std::max(metrics.max_abs, abs_error);
        metrics.max_rel = std::max(metrics.max_rel, rel_error);
        if (abs_error > tolerance) {
            ++metrics.mismatch_count;
        }
    }
    return metrics;
}

// 按值数组和索引数组的实际元素数量统计两种存储格式的空间开销。
size_t coo_storage_bytes(const Problem& problem) {
    return problem.coo.values.size() * (sizeof(float) + 2 * sizeof(uint32_t));
}

size_t csr_storage_bytes(const Problem& problem) {
    return problem.csr.values.size() * (sizeof(float) + sizeof(uint32_t)) +
           problem.csr.row_ptr.size() * sizeof(uint32_t);
}

void print_problem_summary(const Problem& problem) {
    const double density = static_cast<double>(problem.coo.values.size()) /
                           (static_cast<double>(problem.config.rows) * problem.config.cols);
    const size_t coo_bytes = coo_storage_bytes(problem);
    const size_t csr_bytes = csr_storage_bytes(problem);
    const double reduction = coo_bytes == 0 ? 0.0 :
        100.0 * static_cast<double>(coo_bytes - csr_bytes) / coo_bytes;

    std::cout << "Problem: " << problem.config.rows << " x " << problem.config.cols
              << ", nnz=" << problem.coo.values.size()
              << ", nnz/row=" << problem.config.nnz_per_row
              << ", density=" << std::fixed << std::setprecision(4) << density * 100.0 << "%\n"
              << "Storage: COO=" << coo_bytes << " B, CSR=" << csr_bytes << " B"
              << ", CSR reduction=" << std::setprecision(2) << reduction << "%\n"
              << "Partition: COO=" << problem.config.block_dim
              << " balanced nnz blocks; CSR=" << problem.config.block_dim
              << " row blocks\n" << std::defaultfloat;
}

void print_output_sample(const std::vector<float>& y,
                         const std::vector<float>& reference,
                         size_t count) {
    const size_t limit = std::min({count, y.size(), reference.size()});
    std::cout << "sample [row: actual, reference]\n";
    for (size_t i = 0; i < limit; ++i) {
        std::cout << "  " << i << ": " << y[i] << ", " << reference[i] << '\n';
    }
}

}  // namespace spmv


In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/src/main_cpu.cpp
#include "spmv_cpu.h"

#include <iomanip>
#include <iostream>
#include <stdexcept>
#include <string>

namespace {

void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "  --rows <n>          matrix row count, default: 2048\n"
              << "  --cols <n>          matrix column count, default: 2048\n"
              << "  --nnz-per-row <n>   exactly n random non-zeros per row, default: 32\n"
              << "  --block-dim <n>     logical AI Core count, default: 32\n"
              << "  --seed <n>          random seed, default: 20260717\n"
              << "  --warmup <n>        warm-up rounds, default: 5\n"
              << "  --repeat <n>        timed rounds, default: 30\n"
              << "  --print-output      print the first eight CSR values\n";
}

// 解析矩阵规模、逻辑核数、预热次数和正式运行次数等实验参数。
spmv::Config parse_args(int argc, char** argv, bool& print_output) {
    spmv::Config config;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto value = [&](const std::string& option) -> const char* {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + option);
            return argv[++i];
        };
        if (arg == "--rows") config.rows = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--cols") config.cols = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--nnz-per-row") config.nnz_per_row = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--block-dim") config.block_dim = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--seed") config.seed = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--warmup") config.warmup = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--repeat") config.repeat = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--print-output") print_output = true;
        else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown option: " + arg);
        }
    }
    return config;
}

void print_metrics(const char* name, const spmv::TimedResult& result,
                   const spmv::ErrorMetrics& error) {
    std::cout << std::left << std::setw(8) << name
              << std::right << std::setw(14) << std::fixed << std::setprecision(2) << result.elapsed_us
              << std::setw(16) << std::scientific << error.max_abs
              << std::setw(16) << error.max_rel
              << std::setw(12) << std::defaultfloat << error.mismatch_count << '\n';
}

}  // namespace

int main(int argc, char** argv) {
    try {
        bool print_output = false;
        const spmv::Config config = parse_args(argc, argv, print_output);
        spmv::validate_config(config, false);
        // 准备COO、CSR输入数据以及CPU串行参考结果。
        const spmv::Problem problem = spmv::make_problem(config);
        const auto reference = spmv::spmv_reference(problem);

        spmv::print_problem_summary(problem);
        // 分别运行两种分块模拟实现，并按统一口径统计平均耗时。
        const auto coo = spmv::benchmark_coo_cpu(problem, config.warmup, config.repeat);
        const auto csr = spmv::benchmark_csr_cpu(problem, config.warmup, config.repeat);
        // 将两种输出与参考结果比较，形成正确性判定指标。
        const auto coo_error = spmv::compare_vectors(coo.y, reference);
        const auto csr_error = spmv::compare_vectors(csr.y, reference);

        std::cout << std::left << std::setw(8) << "format"
                  << std::right << std::setw(14) << "avg_us"
                  << std::setw(16) << "max_abs"
                  << std::setw(16) << "max_rel"
                  << std::setw(12) << "mismatch" << '\n';
        print_metrics("COO", coo, coo_error);
        print_metrics("CSR", csr, csr_error);
        std::cout << "CPU simulator COO/CSR time ratio: " << std::fixed << std::setprecision(3)
                  << (coo.elapsed_us / csr.elapsed_us) << "x\n";
        std::cout << "Note: CPU times validate the partition/reduction logic only; use spmv_ascend_demo "
                     "for the NPU performance comparison.\n";
        if (print_output) spmv::print_output_sample(csr.y, reference);
        return (coo_error.mismatch_count == 0 && csr_error.mismatch_count == 0) ? 0 : 2;
    } catch (const std::exception& error) {
        std::cerr << "error: " << error.what() << '\n';
        usage(argv[0]);
        return 1;
    }
}


In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/host_launch/spmv_npu_main.cpp
#include <acl/acl.h>
#include <aclrtlaunch_coo_reduce_partial.h>
#include <aclrtlaunch_coo_spmv_partial.h>
#include <aclrtlaunch_csr_spmv.h>

#include "spmv_cpu.h"

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <cstdlib>
#include <iomanip>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>

// 统一检查ACL接口返回值，并将运行时错误转换为可读异常。
#define ACL_CHECK(expression)                                                                  \
    do {                                                                                       \
        const aclError acl_status = (expression);                                              \
        if (acl_status != ACL_SUCCESS) {                                                       \
            throw std::runtime_error(std::string("ACL failure: ") + #expression +             \
                                     ", code=" + std::to_string(static_cast<int>(acl_status))); \
        }                                                                                      \
    } while (0)

namespace {

class Timer {
public:
    Timer() : begin_(std::chrono::high_resolution_clock::now()) {}
    double elapsed_us() const {
        return std::chrono::duration<double, std::micro>(
            std::chrono::high_resolution_clock::now() - begin_).count();
    }

private:
    std::chrono::high_resolution_clock::time_point begin_;
};

struct NpuConfig {
    spmv::Config problem;
    int32_t device = 0;
    bool print_output = false;
};

// 集中管理Device内存、执行流和ACL状态，异常退出时仍能按序释放资源。
struct DeviceResources {
    void* values = nullptr;
    void* coo_cols = nullptr;
    void* coo_rows = nullptr;
    void* csr_cols = nullptr;
    void* csr_row_ptr = nullptr;
    void* x = nullptr;
    void* y = nullptr;
    void* coo_partial = nullptr;
    aclrtStream stream = nullptr;
    int32_t device = 0;
    bool device_set = false;
    bool acl_initialized = false;

    void release() noexcept {
        if (stream != nullptr) aclrtDestroyStream(stream);
        if (coo_partial != nullptr) aclrtFree(coo_partial);
        if (y != nullptr) aclrtFree(y);
        if (x != nullptr) aclrtFree(x);
        if (csr_row_ptr != nullptr) aclrtFree(csr_row_ptr);
        if (csr_cols != nullptr) aclrtFree(csr_cols);
        if (coo_rows != nullptr) aclrtFree(coo_rows);
        if (coo_cols != nullptr) aclrtFree(coo_cols);
        if (values != nullptr) aclrtFree(values);
        if (device_set) aclrtResetDevice(device);
        if (acl_initialized) aclFinalize();
    }
};

void usage(const char* argv0) {
    std::cout << "Usage: " << argv0 << " [options]\n"
              << "  --device <id>        ACL device id, default: 0\n"
              << "  --rows <n>           matrix row count, default: 2048\n"
              << "  --cols <n>           matrix column count, default: 2048\n"
              << "  --nnz-per-row <n>    8/16/32/64 non-zeros per row, default: 32\n"
              << "  --block-dim <n>      AI Core launch blockDim, default: 32\n"
              << "  --seed <n>           random seed, default: 20260717\n"
              << "  --warmup <n>         warm-up rounds per format, default: 5\n"
              << "  --repeat <n>         timed rounds per format, default: 30\n"
              << "  --print-output       print first eight CSR output values\n"
              << "\nAll dimensions and nnz-per-row must be multiples of 8.\n";
}

// 解析设备编号、矩阵参数、blockDim及性能测试轮次。
NpuConfig parse_args(int argc, char** argv) {
    NpuConfig config;
    for (int i = 1; i < argc; ++i) {
        const std::string arg = argv[i];
        auto value = [&](const std::string& option) -> const char* {
            if (i + 1 >= argc) throw std::invalid_argument("missing value after " + option);
            return argv[++i];
        };
        if (arg == "--device") config.device = std::stoi(value(arg));
        else if (arg == "--rows") config.problem.rows = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--cols") config.problem.cols = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--nnz-per-row") config.problem.nnz_per_row = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--block-dim") config.problem.block_dim = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--seed") config.problem.seed = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--warmup") config.problem.warmup = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--repeat") config.problem.repeat = static_cast<uint32_t>(std::stoul(value(arg)));
        else if (arg == "--print-output") config.print_output = true;
        else if (arg == "-h" || arg == "--help") {
            usage(argv[0]);
            std::exit(0);
        } else {
            throw std::invalid_argument("unknown option: " + arg);
        }
    }
    return config;
}

// 将Host数组补齐至32字节对齐所需的元素数量，满足DataCopy约束。
template <typename T>
std::vector<T> pad_to_alignment(const std::vector<T>& source, T padding_value) {
    std::vector<T> padded = source;
    padded.resize(spmv::align_up(static_cast<uint32_t>(source.size())), padding_value);
    return padded;
}

void allocate(void** address, size_t bytes) {
    ACL_CHECK(aclrtMalloc(address, bytes, ACL_MEM_MALLOC_HUGE_FIRST));
}

void copy_host_to_device(void* destination, const void* source, size_t bytes) {
    ACL_CHECK(aclrtMemcpy(destination, bytes, source, bytes, ACL_MEMCPY_HOST_TO_DEVICE));
}

// COO采用两阶段执行：先生成每个AI Core的部分向量，再完成跨核归约。
double launch_coo(const NpuConfig& config, const DeviceResources& resources,
                  uint32_t nnz, uint32_t output_stride) {
    Timer timer;
    ACLRT_LAUNCH_KERNEL(coo_spmv_partial)(config.problem.block_dim, resources.stream,
                                          resources.values, resources.coo_cols, resources.coo_rows,
                                          resources.x, resources.coo_partial,
                                          config.problem.rows, config.problem.cols, nnz, output_stride);
    ACLRT_LAUNCH_KERNEL(coo_reduce_partial)(config.problem.block_dim, resources.stream,
                                            resources.coo_partial, resources.y,
                                            config.problem.rows, output_stride);
    ACL_CHECK(aclrtSynchronizeStream(resources.stream));
    return timer.elapsed_us();
}

// CSR按互不重叠的行区间直接生成输出向量。
double launch_csr(const NpuConfig& config, const DeviceResources& resources,
                  uint32_t nnz, uint32_t output_stride) {
    Timer timer;
    ACLRT_LAUNCH_KERNEL(csr_spmv)(config.problem.block_dim, resources.stream,
                                  resources.values, resources.csr_cols, resources.csr_row_ptr,
                                  resources.x, resources.y,
                                  config.problem.rows, config.problem.cols, nnz, output_stride);
    ACL_CHECK(aclrtSynchronizeStream(resources.stream));
    return timer.elapsed_us();
}

// 预热结束后重复启动算子，以平均值降低首次运行和系统抖动的影响。
double benchmark_coo(const NpuConfig& config, const DeviceResources& resources,
                     uint32_t nnz, uint32_t output_stride) {
    for (uint32_t i = 0; i < config.problem.warmup; ++i) {
        (void)launch_coo(config, resources, nnz, output_stride);
    }
    double total_us = 0.0;
    for (uint32_t i = 0; i < config.problem.repeat; ++i) {
        total_us += launch_coo(config, resources, nnz, output_stride);
    }
    return total_us / config.problem.repeat;
}

double benchmark_csr(const NpuConfig& config, const DeviceResources& resources,
                     uint32_t nnz, uint32_t output_stride) {
    for (uint32_t i = 0; i < config.problem.warmup; ++i) {
        (void)launch_csr(config, resources, nnz, output_stride);
    }
    double total_us = 0.0;
    for (uint32_t i = 0; i < config.problem.repeat; ++i) {
        total_us += launch_csr(config, resources, nnz, output_stride);
    }
    return total_us / config.problem.repeat;
}

void print_result_row(const char* name, double elapsed_us, const spmv::ErrorMetrics& error) {
    std::cout << std::left << std::setw(8) << name
              << std::right << std::setw(14) << std::fixed << std::setprecision(2) << elapsed_us
              << std::setw(16) << std::scientific << error.max_abs
              << std::setw(16) << error.max_rel
              << std::setw(12) << std::defaultfloat << error.mismatch_count << '\n';
}

}  // namespace

int main(int argc, char** argv) {
    DeviceResources resources;
    try {
        const NpuConfig config = parse_args(argc, argv);
        spmv::validate_config(config.problem, true);
        // Host侧生成统一输入，并提前计算CPU串行参考结果。
        const spmv::Problem problem = spmv::make_problem(config.problem);
        const std::vector<float> reference = spmv::spmv_reference(problem);
        const uint32_t nnz = static_cast<uint32_t>(problem.coo.values.size());
        const uint32_t output_stride = spmv::align_up(config.problem.rows);

        const std::vector<float> padded_x = pad_to_alignment(problem.x, 0.0F);
        const std::vector<uint32_t> padded_row_ptr = pad_to_alignment(
            problem.csr.row_ptr, problem.csr.row_ptr.back());

        spmv::print_problem_summary(problem);
        // 初始化ACL运行时，选择逻辑设备并创建算子执行流。
        ACL_CHECK(aclInit(nullptr));
        resources.acl_initialized = true;
        ACL_CHECK(aclrtSetDevice(config.device));
        resources.device = config.device;
        resources.device_set = true;
        ACL_CHECK(aclrtCreateStream(&resources.stream));

        // 为COO、CSR、输入向量、输出向量和COO部分结果分配Device内存。
        allocate(&resources.values, problem.coo.values.size() * sizeof(float));
        allocate(&resources.coo_cols, problem.coo.cols.size() * sizeof(uint32_t));
        allocate(&resources.coo_rows, problem.coo.rows.size() * sizeof(uint32_t));
        allocate(&resources.csr_cols, problem.csr.col_indices.size() * sizeof(uint32_t));
        allocate(&resources.csr_row_ptr, padded_row_ptr.size() * sizeof(uint32_t));
        allocate(&resources.x, padded_x.size() * sizeof(float));
        allocate(&resources.y, static_cast<size_t>(output_stride) * sizeof(float));
        allocate(&resources.coo_partial,
                 static_cast<size_t>(config.problem.block_dim) * output_stride * sizeof(float));

        // 将两种稀疏格式共享的数据及各自的索引结构复制到Device侧。
        copy_host_to_device(resources.values, problem.coo.values.data(),
                            problem.coo.values.size() * sizeof(float));
        copy_host_to_device(resources.coo_cols, problem.coo.cols.data(),
                            problem.coo.cols.size() * sizeof(uint32_t));
        copy_host_to_device(resources.coo_rows, problem.coo.rows.data(),
                            problem.coo.rows.size() * sizeof(uint32_t));
        copy_host_to_device(resources.csr_cols, problem.csr.col_indices.data(),
                            problem.csr.col_indices.size() * sizeof(uint32_t));
        copy_host_to_device(resources.csr_row_ptr, padded_row_ptr.data(),
                            padded_row_ptr.size() * sizeof(uint32_t));
        copy_host_to_device(resources.x, padded_x.data(), padded_x.size() * sizeof(float));

        // 按相同预热和重复次数运行两种算子，并将输出复制回Host侧。
        const double coo_us = benchmark_coo(config, resources, nnz, output_stride);
        std::vector<float> coo_y(config.problem.rows);
        ACL_CHECK(aclrtMemcpy(coo_y.data(), coo_y.size() * sizeof(float), resources.y,
                              coo_y.size() * sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST));

        const double csr_us = benchmark_csr(config, resources, nnz, output_stride);
        std::vector<float> csr_y(config.problem.rows);
        ACL_CHECK(aclrtMemcpy(csr_y.data(), csr_y.size() * sizeof(float), resources.y,
                              csr_y.size() * sizeof(float), ACL_MEMCPY_DEVICE_TO_HOST));

        // 使用统一误差指标验证结果，并输出COO与CSR的性能对比。
        const auto coo_error = spmv::compare_vectors(coo_y, reference);
        const auto csr_error = spmv::compare_vectors(csr_y, reference);
        std::cout << std::left << std::setw(8) << "format"
                  << std::right << std::setw(14) << "avg_us"
                  << std::setw(16) << "max_abs"
                  << std::setw(16) << "max_rel"
                  << std::setw(12) << "mismatch" << '\n';
        print_result_row("COO", coo_us, coo_error);
        print_result_row("CSR", csr_us, csr_error);
        std::cout << "NPU COO/CSR total-time ratio: " << std::fixed << std::setprecision(3)
                  << (coo_us / csr_us) << "x\n";
        std::cout << "COO total = partial SpMV + device reduction; CSR total = row-owned SpMV.\n";
        if (config.print_output) spmv::print_output_sample(csr_y, reference);

        resources.release();
        return (coo_error.mismatch_count == 0 && csr_error.mismatch_count == 0) ? 0 : 2;
    } catch (const std::exception& error) {
        resources.release();
        std::cerr << "error: " << error.what() << '\n';
        usage(argv[0]);
        return 1;
    }
}


### 2.3 工程公共文件检查

公共头文件、CPU参考实现和Host侧启动代码写入完成后，先检查关键文件是否已经生成。检查结果均为`OK`时，说明第2节的工程骨架已经就绪。

In [ ]:
from pathlib import Path

required_files = [
    "src/02.03_extra_spmv_coo_csr_storage_format/include/spmv_common.h",
    "src/02.03_extra_spmv_coo_csr_storage_format/include/spmv_cpu.h",
    "src/02.03_extra_spmv_coo_csr_storage_format/src/spmv_cpu.cpp",
    "src/02.03_extra_spmv_coo_csr_storage_format/src/main_cpu.cpp",
    "src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/host_launch/spmv_npu_main.cpp",
]

for file in required_files:
    path = Path(file)
    print(f"{file}: {'OK' if path.exists() else 'MISSING'}")


---
## 3. 问题分析

本节分析稀疏矩阵向量乘法的输入输出、COO与CSR数据布局、存储开销、多核分块策略、跨核归约方法和实验参数设置。后续核函数开发将围绕本节内容展开。

### 3.1 输入输出与计算公式

本实验实现单精度浮点稀疏矩阵向量乘法。输入由稀疏矩阵$A$与稠密向量$x$组成，输出为稠密向量$y$。矩阵行数为`rows`，列数为`cols`，每行非零元数量固定为`nnzPerRow`，因此总非零元数量为：

$$
nnz=rows\times nnzPerRow
$$

第$r$个输出元素为该矩阵行所有非零元与对应向量元素乘积之和：

$$
y_r=\sum_{j=0}^{nnzPerRow-1}A_{r,col_{r,j}}\times x_{col_{r,j}}
$$

Host侧以固定随机种子生成输入向量和每行互不重复的随机列索引，并按行序保存非零元。CPU参考实现使用COO数组顺序累加得到参考结果。NPU实现分别运行COO与CSR两种并行方案，并将结果与同一参考结果逐元素比较。

### 3.2 COO数据布局

COO格式使用三个等长数组保存每个非零元的数值、行索引和列索引：

$$
values[k]=A_{rows[k],cols[k]},\quad0\le k<nnz
$$

工程中的`CooMatrix`对应`values`、`rows`和`cols`三个数组。每个非零元占用一个`float`数值和两个`uint32_t`索引，因此COO格式的理论存储开销为：

$$
S_{COO}=nnz\times(4+4+4)\text{字节}
$$

COO格式便于按非零元数量均衡划分任务。由于不同任务处理的非零元可能属于同一矩阵行，多个任务不能直接写入同一个输出元素。当前实现为每个逻辑任务分配独立的部分结果向量，随后启动第二个核函数进行按行归约。

### 3.3 CSR数据布局

CSR格式使用`values`、`col_indices`和`row_ptr`三个数组。`values`与`col_indices`保存所有非零元的数值和列索引，`row_ptr`保存每行非零元区间的起始位置。第$r$行对应的非零元区间为：

$$
[rowPtr_r,rowPtr_{r+1})
$$

工程中的`CsrMatrix`使用`row_ptr`、`col_indices`和`values`保存该结构。由于行索引由行指针隐式表达，CSR格式的理论存储开销为：

$$
S_{CSR}=nnz\times(4+4)+4\times(rows+1)\text{字节}
$$

当每行非零元数量大于一时，CSR通常比COO节省行索引存储。更重要的是，CSR可以将连续的矩阵行分配给不同逻辑任务，每个输出元素仅由一个任务写入，不需要跨核归约。

### 3.4 分块策略与数据对齐

COO核函数按非零元数量划分任务。设核函数启动任务数为`blockDim`，每个任务的对齐后非零元数量为：

$$
nnzPerCore=AlignUp\left(\left\lceil\frac{nnz}{blockDim}\right\rceil,8\right)
$$

第`coreId`个任务处理连续非零元区间。每个任务在UB中累加一个长度为`outputStride`的部分结果向量，并写入`partialY`中专属的行区间。`outputStride`为按8元素对齐后的输出长度。COO归约核函数再按行划分任务，每个任务累加所有部分结果向量的对应区间。

CSR核函数按连续矩阵行划分任务。每个任务读取完整的`row_ptr`与输入向量，并计算自己负责的矩阵行。由于行区间互不重叠，输出向量中的每个元素只有一个写入者。

Ascend C的`DataCopy`要求传输字节数满足32字节对齐。为保证数值和索引传输均满足该条件，NPU版本要求`rows`、`cols`与`nnzPerRow`均为8的倍数。当前静态LocalTensor容量还要求`rows`和`cols`不超过4096。

### 3.5 COO与CSR并行计算流程

本实验的COO方案由两个Device侧核函数组成。第一阶段`coo_spmv_partial`按非零元数量划分任务，计算每个任务独立的部分输出向量。主要数据流为：

```text
GM中的values、rows、cols、x→UB中的局部数据与partialLocal→GM中的partialY
```

第二阶段`coo_reduce_partial`按输出行划分任务，将所有任务的部分输出向量对应元素求和：

```text
GM中的partialY→UB中的sumLocal→GM中的y
```

CSR方案由`csr_spmv`完成。每个任务拥有连续的矩阵行和对应输出区间，直接得到最终结果：

```text
GM中的values、colIndices、rowPtr、x→UB中的局部数据与yLocal→GM中的y
```

因此，COO方案的并行粒度由非零元数量决定，但需要额外的部分结果存储与归约；CSR方案的并行粒度由矩阵行决定，避免了跨核输出冲突。

### 3.6 Host侧与Device侧协同执行

Host侧首先生成同时包含COO与CSR表示的同一个稀疏矩阵，并计算CPU参考结果。NPU执行时，Host侧申请并初始化Device内存，依次运行COO的两个核函数和CSR核函数，再将两个输出向量回拷至Host侧进行误差统计。

COO与CSR均执行预热和重复计时。COO计时覆盖部分SpMV核函数、归约核函数与流同步；CSR计时覆盖按行SpMV核函数与流同步。两者的平均用时可以用于观察存储格式、分块方式和归约开销带来的差异，但性能结论应以同一昇腾处理器、相同CANN版本和稳定运行条件下的多次测量为准。

### 3.7 实验参数设置

本实验默认矩阵规模为2048行和2048列，每行包含32个随机非零元，默认启动32个逻辑计算任务，预热次数为5，重复运行次数为30。随机种子默认为20260717，数据类型为`float`。默认总非零元数量为65536，矩阵密度为1.5625%。

In [ ]:
ROWS = 2048
COLS = 2048
NNZ_PER_ROW = 32
BLOCK_DIM = 32
WARMUP = 5
REPEAT = 30
SEED = 20260717

NNZ = ROWS * NNZ_PER_ROW
DENSITY = NNZ / (ROWS * COLS)
COO_BYTES = NNZ * (4 + 4 + 4)
CSR_BYTES = NNZ * (4 + 4) + (ROWS + 1) * 4

print("rows:", ROWS)
print("cols:", COLS)
print("nnzPerRow:", NNZ_PER_ROW)
print("nnz:", NNZ)
print("density:", f"{DENSITY * 100:.4f}%")
print("blockDim:", BLOCK_DIM)
print("COO bytes:", COO_BYTES)
print("CSR bytes:", CSR_BYTES)
print("CSR storage reduction:", f"{(COO_BYTES - CSR_BYTES) * 100 / COO_BYTES:.2f}%")


需要注意，`blockDim`表示核函数启动的逻辑任务数，不应超过当前昇腾处理器可用的AI Core数量。COO第一阶段为每个逻辑任务分配一个长度为`outputStride`的部分结果向量，因此增大`blockDim`会增大`partialY`的Device内存占用和第二阶段归约量。CSR方案不使用`partialY`，但行数较少或每行非零元分布不均时，按行划分可能产生负载不均衡。

---
## 4. 核函数开发

本节实现稀疏矩阵向量乘法中的三个Device侧核函数。第一个核函数`coo_spmv_partial`按非零元区间计算COO部分结果；第二个核函数`coo_reduce_partial`按输出行归约所有部分结果；第三个核函数`csr_spmv`按连续矩阵行计算CSR结果。

本实验采用静态LocalTensor编程方式。核函数在UB中显式申请固定容量的LocalTensor，通过`DataCopy`、片上累加和写回操作完成数据处理。

### 4.1 COO分段计算核函数

`coo_spmv_partial`的输入为COO三数组和输入向量`x`，输出为每个逻辑任务独立的部分结果向量`partialY`。每个任务计算一个连续的非零元区间，避免因多个任务更新同一输出行而产生写冲突。

该核函数的主要流程如下：

1. 根据`GetBlockIdx`与`GetBlockNum`获取当前逻辑任务编号和任务总数；
2. 计算当前任务负责的对齐后非零元区间；
3. 将输入向量搬入UB，并将局部部分结果初始化为零；
4. 按256个非零元为一组搬运数值、行索引和列索引；
5. 在UB中执行乘加，将结果累加至对应行；
6. 将当前任务的局部部分结果写入`partialY`专属区间。

In [ ]:
from pathlib import Path

KERNEL_DIR = Path("src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel")
KERNEL_DIR.mkdir(parents=True, exist_ok=True)
print("Kernel directory:", KERNEL_DIR.resolve())


In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/coo_spmv_partial.cpp
// COO SpMV, stage 1: balance non-zero elements across AI Cores.
//
// A COO block may contain entries from the same row as another block.  Directly
// adding into y would therefore race.  Each core writes an independent
// partial-y vector; coo_reduce_partial.cpp owns the second-stage reduction.

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kMaxMatrixDim = 4096;
constexpr uint32_t kNnzTile = 256;
constexpr uint32_t kAlignElements = 8;

__aicore__ inline uint32_t DivUp(uint32_t numerator, uint32_t denominator) {
    return (numerator + denominator - 1) / denominator;
}

__aicore__ inline uint32_t AlignUp(uint32_t value) {
    return DivUp(value, kAlignElements) * kAlignElements;
}
}  // namespace

extern "C" __global__ __aicore__ void coo_spmv_partial(GM_ADDR values,
                                                         GM_ADDR colIndices,
                                                         GM_ADDR rowIndices,
                                                         GM_ADDR x,
                                                         GM_ADDR partialY,
                                                         uint32_t numRows,
                                                         uint32_t numCols,
                                                         uint32_t nnz,
                                                         uint32_t outputStride) {
    InitSocState();
    if (numRows == 0 || numCols == 0 || numRows > kMaxMatrixDim ||
        numCols > kMaxMatrixDim || outputStride > kMaxMatrixDim) {
        return;
    }

    const uint32_t coreId = GetBlockIdx();
    const uint32_t coreCount = GetBlockNum();
    if (coreCount == 0 || coreId >= coreCount) {
        return;
    }

    // 建立GM张量视图，分别访问非零值、行列索引、输入向量和部分结果。
    GlobalTensor<float> valuesGm;
    GlobalTensor<uint32_t> colsGm;
    GlobalTensor<uint32_t> rowsGm;
    GlobalTensor<float> xGm;
    GlobalTensor<float> partialYGm;
    valuesGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(values), nnz);
    colsGm.SetGlobalBuffer(reinterpret_cast<__gm__ uint32_t*>(colIndices), nnz);
    rowsGm.SetGlobalBuffer(reinterpret_cast<__gm__ uint32_t*>(rowIndices), nnz);
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), numCols);
    partialYGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(partialY),
                               coreCount * outputStride);

    // 在UB中保存输入向量、当前核的部分输出以及分块搬入的COO元素。
    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> xLocal = ubAllocator.Alloc<float, kMaxMatrixDim>();
    LocalTensor<float> partialLocal = ubAllocator.Alloc<float, kMaxMatrixDim>();
    LocalTensor<float> valuesLocal = ubAllocator.Alloc<float, kNnzTile>();
    LocalTensor<uint32_t> colsLocal = ubAllocator.Alloc<uint32_t, kNnzTile>();
    LocalTensor<uint32_t> rowsLocal = ubAllocator.Alloc<uint32_t, kNnzTile>();
    xLocal.SetSize(kMaxMatrixDim);
    partialLocal.SetSize(kMaxMatrixDim);
    valuesLocal.SetSize(kNnzTile);
    colsLocal.SetSize(kNnzTile);
    rowsLocal.SetSize(kNnzTile);

    // Host-side validation guarantees 32-byte-aligned transfer sizes.
    DataCopy(xLocal, xGm, numCols);
    for (uint32_t row = 0; row < outputStride; ++row) {
        partialLocal.SetValue(row, 0.0F);
    }

    // 按非零元素数量均衡划分任务，并将边界向上对齐。
    const uint32_t nnzPerCore = AlignUp(DivUp(nnz, coreCount));
    const uint32_t begin = coreId * nnzPerCore;
    const uint32_t end = begin < nnz ? ((begin + nnzPerCore < nnz) ?
        (begin + nnzPerCore) : nnz) : begin;

    // 分批搬运当前核负责的COO元素，完成乘法并按行累加到局部向量。
    for (uint32_t offset = begin; offset < end;) {
        const uint32_t count = (end - offset < kNnzTile) ? end - offset : kNnzTile;
        DataCopy(valuesLocal, valuesGm[offset], count);
        DataCopy(colsLocal, colsGm[offset], count);
        DataCopy(rowsLocal, rowsGm[offset], count);
        PipeBarrier<PIPE_ALL>();

        for (uint32_t i = 0; i < count; ++i) {
            const uint32_t row = rowsLocal.GetValue(i);
            const uint32_t col = colsLocal.GetValue(i);
            if (row < numRows && col < numCols) {
                const float product = valuesLocal.GetValue(i) * xLocal.GetValue(col);
                partialLocal.SetValue(row, partialLocal.GetValue(row) + product);
            }
        }
        offset += count;
    }
    // 每个核写入独立的部分结果区域，避免多个核同时更新同一输出行。
    DataCopy(partialYGm[coreId * outputStride], partialLocal, outputStride);
}


### 4.2 COO部分结果归约核函数

`coo_reduce_partial`读取第一阶段生成的`partialY`，输出最终向量`y`。该核函数按连续输出行划分任务，每个任务只写自己负责的行区间。

该核函数的主要流程如下：

1. 根据当前AI Core编号计算负责处理的输出行区间；
2. 在UB中申请一个累加LocalTensor和一个单任务部分结果LocalTensor；
3. 将当前行区间的累加LocalTensor初始化为零；
4. 依次搬运所有逻辑任务的对应部分结果，并累加至UB；
5. 将当前行区间的归约结果写回GM中的`y`。

In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/coo_reduce_partial.cpp
// COO SpMV, stage 2: one owner core reduces every output-row interval.

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kMaxMatrixDim = 4096;
constexpr uint32_t kAlignElements = 8;

__aicore__ inline uint32_t DivUp(uint32_t numerator, uint32_t denominator) {
    return (numerator + denominator - 1) / denominator;
}

__aicore__ inline uint32_t AlignUp(uint32_t value) {
    return DivUp(value, kAlignElements) * kAlignElements;
}
}  // namespace

extern "C" __global__ __aicore__ void coo_reduce_partial(GM_ADDR partialY,
                                                           GM_ADDR y,
                                                           uint32_t numRows,
                                                           uint32_t outputStride) {
    InitSocState();
    if (numRows == 0 || numRows > kMaxMatrixDim || outputStride > kMaxMatrixDim) {
        return;
    }

    const uint32_t coreId = GetBlockIdx();
    const uint32_t coreCount = GetBlockNum();
    if (coreCount == 0 || coreId >= coreCount) {
        return;
    }
    // 将输出行划分给不同归约核，使每个输出元素仅由一个核写回。
    const uint32_t rowsPerCore = AlignUp(DivUp(numRows, coreCount));
    const uint32_t rowBegin = coreId * rowsPerCore;
    if (rowBegin >= numRows) {
        return;
    }
    const uint32_t rowEnd = (rowBegin + rowsPerCore < numRows) ?
        rowBegin + rowsPerCore : numRows;
    const uint32_t count = rowEnd - rowBegin;

    // 建立部分结果矩阵和最终输出向量的GM视图。
    GlobalTensor<float> partialYGm;
    GlobalTensor<float> yGm;
    partialYGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(partialY),
                               coreCount * outputStride);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), outputStride);

    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> sumLocal = ubAllocator.Alloc<float, kMaxMatrixDim>();
    LocalTensor<float> onePartialLocal = ubAllocator.Alloc<float, kMaxMatrixDim>();
    sumLocal.SetSize(kMaxMatrixDim);
    onePartialLocal.SetSize(kMaxMatrixDim);
    for (uint32_t i = 0; i < count; ++i) {
        sumLocal.SetValue(i, 0.0F);
    }

    // 依次读取所有生产核在本行区间上的部分结果并累加。
    for (uint32_t producer = 0; producer < coreCount; ++producer) {
        DataCopy(onePartialLocal, partialYGm[producer * outputStride + rowBegin], count);
        PipeBarrier<PIPE_ALL>();
        for (uint32_t i = 0; i < count; ++i) {
            sumLocal.SetValue(i, sumLocal.GetValue(i) + onePartialLocal.GetValue(i));
        }
    }
    // 将当前核负责的连续行区间一次性写回最终输出。
    DataCopy(yGm[rowBegin], sumLocal, count);
}


### 4.3 CSR按行计算核函数

`csr_spmv`直接读取CSR三数组和输入向量`x`，输出最终向量`y`。每个任务处理一段连续矩阵行，因此该任务独占对应输出元素，无需额外归约。

该核函数的主要流程如下：

1. 根据当前AI Core编号计算负责处理的矩阵行区间；
2. 将输入向量和行指针搬入UB；
3. 对每一行，根据相邻行指针确定非零元区间；
4. 按256个非零元为一组搬运数值与列索引；
5. 在UB中累加当前行的乘积结果；
6. 将连续行的最终结果写回GM中的`y`。

In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/csr_spmv.cpp
// CSR SpMV: cores own disjoint matrix-row intervals, so each y[row] has one
// writer and no cross-core reduction is required.

#include "kernel_operator.h"

using namespace AscendC;

namespace {
constexpr uint32_t kMaxMatrixDim = 4096;
constexpr uint32_t kRowPtrCapacity = kMaxMatrixDim + 8;
constexpr uint32_t kNnzTile = 256;
constexpr uint32_t kAlignElements = 8;

__aicore__ inline uint32_t DivUp(uint32_t numerator, uint32_t denominator) {
    return (numerator + denominator - 1) / denominator;
}

__aicore__ inline uint32_t AlignUp(uint32_t value) {
    return DivUp(value, kAlignElements) * kAlignElements;
}
}  // namespace

extern "C" __global__ __aicore__ void csr_spmv(GM_ADDR values,
                                                GM_ADDR colIndices,
                                                GM_ADDR rowPtr,
                                                GM_ADDR x,
                                                GM_ADDR y,
                                                uint32_t numRows,
                                                uint32_t numCols,
                                                uint32_t nnz,
                                                uint32_t outputStride) {
    InitSocState();
    if (numRows == 0 || numCols == 0 || numRows > kMaxMatrixDim ||
        numCols > kMaxMatrixDim || outputStride > kMaxMatrixDim) {
        return;
    }

    const uint32_t coreId = GetBlockIdx();
    const uint32_t coreCount = GetBlockNum();
    if (coreCount == 0 || coreId >= coreCount) {
        return;
    }
    // 按连续行区间划分任务，每个输出行由唯一AI Core负责。
    const uint32_t rowsPerCore = AlignUp(DivUp(numRows, coreCount));
    const uint32_t rowBegin = coreId * rowsPerCore;
    if (rowBegin >= numRows) {
        return;
    }
    const uint32_t rowEnd = (rowBegin + rowsPerCore < numRows) ?
        rowBegin + rowsPerCore : numRows;
    const uint32_t localRows = rowEnd - rowBegin;
    const uint32_t rowPtrElements = AlignUp(numRows + 1);

    // 建立CSR数组、输入向量和输出向量的GM张量视图。
    GlobalTensor<float> valuesGm;
    GlobalTensor<uint32_t> colsGm;
    GlobalTensor<uint32_t> rowPtrGm;
    GlobalTensor<float> xGm;
    GlobalTensor<float> yGm;
    valuesGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(values), nnz);
    colsGm.SetGlobalBuffer(reinterpret_cast<__gm__ uint32_t*>(colIndices), nnz);
    rowPtrGm.SetGlobalBuffer(reinterpret_cast<__gm__ uint32_t*>(rowPtr), rowPtrElements);
    xGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(x), numCols);
    yGm.SetGlobalBuffer(reinterpret_cast<__gm__ float*>(y), outputStride);

    // 在UB中缓存输入向量、行指针以及当前批次的非零值和列索引。
    LocalMemAllocator<AscendC::Hardware::UB> ubAllocator;
    LocalTensor<float> xLocal = ubAllocator.Alloc<float, kMaxMatrixDim>();
    LocalTensor<uint32_t> rowPtrLocal = ubAllocator.Alloc<uint32_t, kRowPtrCapacity>();
    LocalTensor<float> yLocal = ubAllocator.Alloc<float, kMaxMatrixDim>();
    LocalTensor<float> valuesLocal = ubAllocator.Alloc<float, kNnzTile>();
    LocalTensor<uint32_t> colsLocal = ubAllocator.Alloc<uint32_t, kNnzTile>();
    xLocal.SetSize(kMaxMatrixDim);
    rowPtrLocal.SetSize(kRowPtrCapacity);
    yLocal.SetSize(kMaxMatrixDim);
    valuesLocal.SetSize(kNnzTile);
    colsLocal.SetSize(kNnzTile);

    // 输入向量和行指针会被当前核重复访问，因此先整体搬入UB。
    DataCopy(xLocal, xGm, numCols);
    DataCopy(rowPtrLocal, rowPtrGm, rowPtrElements);
    PipeBarrier<PIPE_ALL>();

    // 逐行读取rowPtr确定非零区间，再分批完成乘法和行内归约。
    for (uint32_t localRow = 0; localRow < localRows; ++localRow) {
        const uint32_t row = rowBegin + localRow;
        float rowSum = 0.0F;
        const uint32_t rowNnzBegin = rowPtrLocal.GetValue(row);
        const uint32_t rowNnzEnd = rowPtrLocal.GetValue(row + 1);
        for (uint32_t offset = rowNnzBegin; offset < rowNnzEnd;) {
            const uint32_t count = (rowNnzEnd - offset < kNnzTile) ?
                rowNnzEnd - offset : kNnzTile;
            DataCopy(valuesLocal, valuesGm[offset], count);
            DataCopy(colsLocal, colsGm[offset], count);
            PipeBarrier<PIPE_ALL>();
            for (uint32_t i = 0; i < count; ++i) {
                const uint32_t col = colsLocal.GetValue(i);
                if (col < numCols) {
                    rowSum += valuesLocal.GetValue(i) * xLocal.GetValue(col);
                }
            }
            offset += count;
        }
        yLocal.SetValue(localRow, rowSum);
    }
    // 将当前核计算的连续输出行写回GM，无需跨核结果合并。
    DataCopy(yGm[rowBegin], yLocal, localRows);
}


### 4.4 核函数实现要点

COO分段计算核函数中的`partialY`解决了不同非零元区间可能更新同一输出行的问题。其代价是Device内存中需要保存`blockDim`个部分结果向量，并由第二阶段重复读取。归约核函数以行区间为所有权边界，保证每个最终输出元素只有一个写入者。

CSR按行计算核函数直接利用`row_ptr`界定每行非零元范围。相同矩阵行的非零元由同一任务处理，因而不需要原子操作或第二阶段归约。该实现同时体现了稀疏数据结构中索引组织方式对并行写冲突和数据搬运模式的影响。

三个核函数均使用容量为4096的静态LocalTensor处理向量、部分结果或行结果，使用容量为256的静态LocalTensor分块搬运非零元数据。Host侧参数检查保证所有`DataCopy`长度满足对齐约束。

In [ ]:
from pathlib import Path

kernel_files = [
    Path("src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/coo_spmv_partial.cpp"),
    Path("src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/coo_reduce_partial.cpp"),
    Path("src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/csr_spmv.cpp"),
]

for file in kernel_files:
    print(file)
    print("exists:", file.exists(), "size:", file.stat().st_size if file.exists() else 0)


---
## 5. 结果验证与性能分析

核函数开发完成后，本节按照Host侧输入数据和CPU参考结果准备、工程构建、算子运行、结果验证与性能分析五个环节组织实验。Device侧分别执行COO的分段计算与归约，以及CSR的按行计算；Host侧负责完成问题构造、Device内存管理、核函数启动、结果回拷和指标统计。

本节按照完整工程流程写入CMake构建配置和运行脚本，并给出CPU参考版本和NPU验证命令。运行本Notebook后，`src/02.03_extra_spmv_coo_csr_storage_format`目录下将具备可直接编译运行的完整工程文件。

### 5.1 数据准备

Host侧根据`rows`、`cols`、`nnz_per_row`和随机种子构造实验问题。程序为每个矩阵行生成数量固定且互不重复的列索引，并同时建立COO与CSR两种表示，使两种格式对应同一个稀疏矩阵和同一个输入向量。

`spmv_reference`按照COO数组顺序累加，生成输出向量的CPU参考结果。CPU演示程序还模拟COO按非零元分段加归约和CSR按行计算，用于验证两种任务划分的结果一致性。NPU运行完成后，Host侧将Device结果回拷到主机，并与该参考结果逐元素比较。

在构建工程前，先检查Notebook已经写入的关键源码文件。若以下文件均存在，说明本Notebook具备从源码写入到编译运行的基本工程结构。

In [ ]:
from pathlib import Path

required_files = [
    "src/02.03_extra_spmv_coo_csr_storage_format/include/spmv_common.h",
    "src/02.03_extra_spmv_coo_csr_storage_format/include/spmv_cpu.h",
    "src/02.03_extra_spmv_coo_csr_storage_format/src/spmv_cpu.cpp",
    "src/02.03_extra_spmv_coo_csr_storage_format/src/main_cpu.cpp",
    "src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/coo_spmv_partial.cpp",
    "src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/coo_reduce_partial.cpp",
    "src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/op_kernel/csr_spmv.cpp",
    "src/02.03_extra_spmv_coo_csr_storage_format/ascend_ops/host_launch/spmv_npu_main.cpp",
]

for file in required_files:
    path = Path(file)
    print(f"{file}: {'OK' if path.exists() else 'MISSING'}")


### 5.2 工程构建

`CMakeLists.txt`用于组织CPU参考程序、Device侧Ascend C核函数和Host侧调用程序的编译。CPU版本不依赖NPU，用于验证问题构造、参考计算和任务划分逻辑；NPU版本需要引入CANN提供的Ascend C编译配置，并链接ACL运行库。

运行NPU构建脚本时，脚本会加载CANN环境，传入昇腾处理器型号、运行模式和CANN安装路径，再完成核函数库与Host侧可执行程序的构建。

In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/CMakeLists.txt
cmake_minimum_required(VERSION 3.16)
project(ascendc_spmv_storage_formats LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT CMAKE_BUILD_TYPE)
  set(CMAKE_BUILD_TYPE Release CACHE STRING "Build type" FORCE)
endif()

option(BUILD_ASCEND "Build the Ascend C NPU implementation" OFF)

add_library(spmv_cpu
    src/spmv_cpu.cpp
)
target_include_directories(spmv_cpu PUBLIC include)
target_compile_options(spmv_cpu PRIVATE -Wall -Wextra -Wpedantic)

add_executable(spmv_cpu_demo src/main_cpu.cpp)
target_link_libraries(spmv_cpu_demo PRIVATE spmv_cpu)
target_compile_options(spmv_cpu_demo PRIVATE -Wall -Wextra -Wpedantic)

if(BUILD_ASCEND)
  set(RUN_MODE "npu" CACHE STRING "Ascend C run mode: npu/cpu/sim")
  set(SOC_VERSION "ascend910b3" CACHE STRING "Ascend SOC version")
  set(ASCEND_CANN_PATH "$ENV{ASCEND_INSTALL_PATH}" CACHE PATH "CANN installation path")
  if(NOT ASCEND_CANN_PATH)
    set(ASCEND_CANN_PATH "/usr/local/Ascend/ascend-toolkit/latest" CACHE PATH "CANN installation path" FORCE)
  endif()
  set(ASCEND_CANN_PACKAGE_PATH "${ASCEND_CANN_PATH}" CACHE PATH "CANN package path" FORCE)
  set(CMAKE_INSTALL_PREFIX "${CMAKE_BINARY_DIR}/out" CACHE PATH "Ascend C install output" FORCE)

  if(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tools/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/compiler/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  elseif(EXISTS "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
    set(ASCENDC_CMAKE_FILE "${ASCEND_CANN_PACKAGE_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake")
  else()
    message(FATAL_ERROR "Cannot find ascendc.cmake under ${ASCEND_CANN_PACKAGE_PATH}. Set ASCEND_CANN_PATH or ASCEND_INSTALL_PATH.")
  endif()

  message(STATUS "ASCEND_CANN_PACKAGE_PATH=${ASCEND_CANN_PACKAGE_PATH}")
  message(STATUS "SOC_VERSION=${SOC_VERSION}")
  include("${ASCENDC_CMAKE_FILE}")

  ascendc_library(spmv_kernels STATIC
      ascend_ops/op_kernel/coo_spmv_partial.cpp
      ascend_ops/op_kernel/coo_reduce_partial.cpp
      ascend_ops/op_kernel/csr_spmv.cpp
  )
  ascendc_compile_definitions(spmv_kernels PRIVATE -DASCENDC_DUMP=0)

  add_executable(spmv_ascend_demo
      ascend_ops/host_launch/spmv_npu_main.cpp
  )
  target_include_directories(spmv_ascend_demo PRIVATE
      include
      ${ASCEND_CANN_PACKAGE_PATH}/include
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/include
      ${CMAKE_INSTALL_PREFIX}/include/spmv_kernels
      ${CMAKE_BINARY_DIR}/out/include/spmv_kernels
  )
  target_link_directories(spmv_ascend_demo PRIVATE
      ${ASCEND_CANN_PACKAGE_PATH}/lib64
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64/stub
      ${ASCEND_CANN_PACKAGE_PATH}/runtime/lib64
  )
  target_link_libraries(spmv_ascend_demo PRIVATE spmv_kernels spmv_cpu ascendcl)
  add_dependencies(spmv_ascend_demo spmv_kernels)
endif()

install(TARGETS spmv_cpu_demo RUNTIME DESTINATION bin)


### 5.3 算子运行

下面写入CPU版本和NPU版本的运行脚本。CPU脚本负责编译并运行两种稀疏格式的Host侧模拟程序；NPU脚本负责加载CANN环境、配置CMake、编译Ascend C核函数和Host侧程序，并依次运行COO与CSR的SpMV算子。

两个版本均支持通过命令行设置矩阵行数、矩阵列数、每行非零元数量、逻辑任务数、随机种子、预热次数和重复次数。NPU版本还支持设置Device编号。

In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/scripts/run_cpu_demo.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build-cpu"

cmake -S "${SCRIPT_DIR}" -B "${BUILD_DIR}" -DCMAKE_BUILD_TYPE=Release
cmake --build "${BUILD_DIR}" -j
"${BUILD_DIR}/spmv_cpu_demo" "$@"


In [ ]:
%%writefile src/02.03_extra_spmv_coo_csr_storage_format/scripts/run_ascend_spmv.sh
#!/usr/bin/env bash
set -euo pipefail

SCRIPT_DIR=$(cd "$(dirname "${BASH_SOURCE[0]}")/.." && pwd)
BUILD_DIR="${SCRIPT_DIR}/build-ascend"
SOC_VERSION="${SOC_VERSION:-ascend910b3}"
ASCEND_PATH="${ASCEND_INSTALL_PATH:-${ASCEND_HOME_PATH:-/usr/local/Ascend/ascend-toolkit/latest}}"

usage() {
  cat <<'EOF'
Usage: scripts/run_ascend_spmv.sh [-a ASCEND_PATH] [-v SOC_VERSION] [-c] [-- demo options]

  -a PATH   CANN installation path
  -v SOC    Ascend SOC version, e.g. ascend910b3 or ascend310p3
  -c        remove the local Ascend build directory first
  -h        show this help

All arguments after -- are passed to spmv_ascend_demo.
Example:
  bash scripts/run_ascend_spmv.sh -a /usr/local/Ascend/ascend-toolkit/latest -v ascend910b3 -- --block-dim 32 --repeat 30
EOF
}

CLEAN=0
while getopts ":a:v:ch" opt; do
  case "${opt}" in
    a) ASCEND_PATH="${OPTARG}" ;;
    v) SOC_VERSION="${OPTARG}" ;;
    c) CLEAN=1 ;;
    h) usage; exit 0 ;;
    \?) echo "Unknown option: -${OPTARG}" >&2; usage; exit 1 ;;
    :) echo "Option -${OPTARG} needs a value" >&2; usage; exit 1 ;;
  esac
done
shift $((OPTIND - 1))
if [[ $# -gt 0 && "$1" == "--" ]]; then shift; fi

if [[ ! -d "${ASCEND_PATH}" ]]; then
  echo "CANN installation does not exist: ${ASCEND_PATH}" >&2
  exit 1
fi
if [[ ${CLEAN} -eq 1 ]]; then rm -rf "${BUILD_DIR}"; fi
if [[ -f "${ASCEND_PATH}/set_env.sh" ]]; then
  # shellcheck disable=SC1090
  source "${ASCEND_PATH}/set_env.sh"
fi

# CANN 8.3 can expose tikcpp/ and ccec_compiler/ directly below the toolkit,
# while some ascendc.cmake releases still look below ascendc_devkit/.  Mirror
# the completed course project's compatibility view locally instead of
# changing the system installation.
CANN_PATH_FOR_CMAKE="${ASCEND_PATH}"
ASCENDC_NEW_LAYOUT="${ASCEND_PATH}/tikcpp/ascendc_kernel_cmake/ascendc.cmake"
BISHENG_REAL="${ASCEND_PATH}/ccec_compiler/bin/bisheng"
BISHENG_EXPECTED="${ASCEND_PATH}/ascendc_devkit/ccec_compiler/bin/bisheng"
if [[ -f "${ASCENDC_NEW_LAYOUT}" && -x "${BISHENG_REAL}" && ! -x "${BISHENG_EXPECTED}" ]]; then
  CANN_PATH_FOR_CMAKE="${BUILD_DIR}/cann_package_compat"
  mkdir -p "${CANN_PATH_FOR_CMAKE}/ascendc_devkit"
  for entry in "${ASCEND_PATH}"/*; do
    [[ -e "${entry}" ]] || continue
    name=$(basename "${entry}")
    [[ "${name}" == "ascendc_devkit" ]] && continue
    ln -sfn "${entry}" "${CANN_PATH_FOR_CMAKE}/${name}"
  done
  if [[ -d "${ASCEND_PATH}/ascendc_devkit" ]]; then
    for entry in "${ASCEND_PATH}/ascendc_devkit"/*; do
      [[ -e "${entry}" ]] || continue
      name=$(basename "${entry}")
      [[ "${name}" == "ccec_compiler" ]] && continue
      ln -sfn "${entry}" "${CANN_PATH_FOR_CMAKE}/ascendc_devkit/${name}"
    done
  fi
  for component in asc tikcpp ccec_compiler; do
    [[ -e "${ASCEND_PATH}/${component}" ]] || continue
    ln -sfn "${ASCEND_PATH}/${component}" "${CANN_PATH_FOR_CMAKE}/ascendc_devkit/${component}"
  done
  echo "[INFO] Using local CANN compatibility view: ${CANN_PATH_FOR_CMAKE}"
fi

cmake -S "${SCRIPT_DIR}" -B "${BUILD_DIR}" \
  -DCMAKE_BUILD_TYPE=Release \
  -DBUILD_ASCEND=ON \
  -DASCEND_CANN_PATH="${CANN_PATH_FOR_CMAKE}" \
  -DASCEND_CANN_PACKAGE_PATH="${CANN_PATH_FOR_CMAKE}" \
  -DSOC_VERSION="${SOC_VERSION}" \
  -DRUN_MODE=npu
cmake --build "${BUILD_DIR}" -j
"${BUILD_DIR}/spmv_ascend_demo" "$@"


In [ ]:
!chmod +x src/02.03_extra_spmv_coo_csr_storage_format/scripts/run_cpu_demo.sh
!chmod +x src/02.03_extra_spmv_coo_csr_storage_format/scripts/run_ascend_spmv.sh
!find src/02.03_extra_spmv_coo_csr_storage_format -maxdepth 3 -type f | sort


#### 5.3.1 运行CPU版本

先运行CPU版本，确认COO分段加归约与CSR按行计算均能得到与CPU参考结果一致的输出。该命令会完成CPU工程构建和运行。CPU计时仅用于验证分块与归约逻辑，不用于评价NPU算子性能。

In [ ]:
!cd src/02.03_extra_spmv_coo_csr_storage_format && bash scripts/run_cpu_demo.sh --rows 2048 --cols 2048 --nnz-per-row 32 --block-dim 32 | tee results/spmv_cpu_result.txt


#### 5.3.2 运行NPU版本

在已安装CANN且存在NPU的环境中，执行下面命令运行NPU版本。默认矩阵规模为2048行和2048列，每行包含32个非零元，启动32个逻辑计算任务。该命令同时覆盖COO分段计算、COO部分结果归约和CSR按行计算。

In [ ]:
# 在已安装CANN且存在NPU的环境中执行
!cd src/02.03_extra_spmv_coo_csr_storage_format && bash scripts/run_ascend_spmv.sh -- --rows 2048 --cols 2048 --nnz-per-row 32 --block-dim 32 | tee results/spmv_npu_result.txt


### 5.4 结果验证

通过分析`max_abs`和`max_rel`实验指标进行结果验证，其中，`max_abs`表示最大绝对误差，`max_rel`表示最大相对误差。

最大相对误差的计算方式与Host侧代码保持一致：

$$
max\_rel=\max_i\frac{|y_i-ref_i|}{\max(|ref_i|,10^{-12})}
$$

其中，$y_i$为COO或CSR计算结果，$ref_i$为Host侧CPU参考结果。工程将绝对误差大于$10^{-4}$的元素计入`mismatch`。浮点乘加的结合顺序会随COO分段归约和CSR按行累加而改变，因此可能出现微小数值差异。若`mismatch`为0，且`max_abs`与`max_rel`处于较小范围内，可认为相应格式的计算结果可信。

In [ ]:
from pathlib import Path

for path in [
    Path("src/02.03_extra_spmv_coo_csr_storage_format/results/spmv_cpu_result.txt"),
    Path("src/02.03_extra_spmv_coo_csr_storage_format/results/spmv_npu_result.txt"),
]:
    print("=", path)
    if path.exists():
        print(path.read_text(encoding="utf-8", errors="ignore")[:4000])
    else:
        print("not found")


### 5.5 性能分析

通过分析`avg_us`实验指标进行性能分析，该指标表示完成预热后多次运行的平均执行时间，其中，COO的`avg_us`覆盖分段SpMV核函数、部分结果归约核函数和流同步；CSR的`avg_us`覆盖按行SpMV核函数和流同步。程序同时输出两者的时间比值：

$$
timeRatio=\frac{avg\_us_{COO}}{avg\_us_{CSR}}
$$

当该比值大于1时，当前配置下COO方案的总执行时间高于CSR方案；当该比值小于1时，则相反。COO方案的时间开销除非零元乘加外，还包括`partialY`的写入、读取和跨任务归约；CSR方案避免了归约，但其性能会受到按行负载均衡、行内非零元数量和列索引访问局部性的影响。

进行格式性能对比时，应固定矩阵规模、每行非零元数量、逻辑任务数、预热次数和重复次数，并在相同昇腾处理器、相同CANN版本和稳定运行条件下测量。CPU版本的时间用于逻辑验证，NPU版本的时间才用于分析Ascend C算子性能。

---
## 6. 实验总结

本实验按照实验概述、环境准备、算子分析、核函数开发和结果验证与性能分析五个阶段，实现了基于Ascend C静态LocalTensor的COO与CSR稀疏矩阵向量乘法。

* 算子分析明确了SpMV的输入输出、COO与CSR数据布局、两种格式的存储开销和并行任务划分方式。
* COO分段计算核函数按非零元数量均衡划分任务，为每个逻辑任务生成独立部分结果，避免跨核写冲突。
* COO归约核函数按输出行划分任务，将所有部分结果归约为最终输出向量。
* CSR按行计算核函数利用行指针组织非零元区间，使每个输出元素由唯一逻辑任务写入，无需第二阶段归约。
* Host侧程序完成随机问题生成、CPU参考结果计算、Device内存管理、核函数启动、结果回拷和指标统计。
* 结果验证通过最大绝对误差、最大相对误差和超过容差的元素数量，对比Host侧参考结果与Device侧计算结果；性能分析通过平均执行时间比较COO与CSR方案的执行开销。

通过本实验，可以理解稀疏矩阵存储格式对索引开销、数据搬运和并行写冲突的影响，掌握按非零元与按行分块的设计差异，并熟悉Ascend C静态LocalTensor编程中数据搬运、片上计算和Host侧与Device侧协同执行的基本流程。